# 01 - Data Cleaning
This notebook loads, validates, and prepares CTA daily ridership data.

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(".").resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "CTA_-_Ridership_-_Daily_Boarding_Totals_20260526.csv"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

df = pd.read_csv(RAW_PATH)
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)
df["service_date"] = pd.to_datetime(df["service_date"])

# Convert ridership columns to numeric (handles comma-separated values)
for col in ["bus", "rail_boardings", "total_rides"]:
    df[col] = (
        df[col].astype(str).str.replace(",", "", regex=False).str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Validation: does bus + rail approximately match total?
df["components_sum"] = df["bus"] + df["rail_boardings"]
df["difference"] = df["total_rides"] - df["components_sum"]
df["difference_pct_of_total"] = np.where(df["total_rides"] != 0, df["difference"] / df["total_rides"], np.nan)

# Engineered columns
df["year"] = df["service_date"].dt.year
df["month"] = df["service_date"].dt.month
df["month_name"] = df["service_date"].dt.month_name()
df["quarter"] = "Q" + df["service_date"].dt.quarter.astype(str)
df["day_of_week"] = df["service_date"].dt.day_name()
df["day_type_label"] = df["day_type"].map({"W": "Weekday", "A": "Saturday", "U": "Sunday/Holiday"}).fillna("Unknown")
df["bus_share"] = np.where(df["total_rides"] > 0, df["bus"] / df["total_rides"], np.nan)
df["rail_share"] = np.where(df["total_rides"] > 0, df["rail_boardings"] / df["total_rides"], np.nan)

df.head()


In [ ]:

# Export cleaned daily file
df.to_csv(CLEANED_DIR / "cta_ridership_cleaned.csv", index=False)

# Quick quality checks
print("Rows:", len(df))
print("Date range:", df["service_date"].min().date(), "to", df["service_date"].max().date())
print("Max absolute difference (total vs components):", df["difference"].abs().max())
